# CSP(1 point)
In the following code, only two of the function usages are essential. One is `is_valid(self, state)`, which checks if the current state is legal; the other is `is_satisfy(self, state)`, which is to checks if the current board meets the win condition. 
The type of state is [], which stores the tuples(a, b), representing the positions of the queens in it.

In test block, for `Solver`,  `n` indicates the size of the test while `method` indicates which bts function will be used(`bts` or `improving_bts`or`min_conflict`); for method `solve`, `render` indicates whether to use a graphical interface representation.

## Question 1: You should complete the function `bts()`. (0.8 points)
You can only use iterative way, not recursive. Using recursion will incur a **20% penalty**. You may add any function you want. 
**Remember to test the case where N=6**

## Question 2: You should complete the function `improving_bts()` or `min_conflict()`. (0.2 points)
For `improving_bts()`, you can use one or more of the following strategies: Minimum Remaining Value, Least constraining value, and Forward checking. You should have a good performance (within 2s) **when N = 16 without GUI**. 

For `min_conflict()`, you should have a good performance  (within 2s) **when N = 100 without GUI**.

### DDL: April.1st
The practice will be checked in this lab class or the next lab class(before **April.1st**) by teachers or SAs. 
#### What will be tested: 
* That you understand every line of your code, not just copy from somewhere 
* That your program compiles correctly
* Correctness of the program logic 
* That the result is obtained in a reasonable time 

#### Grading: 
* Submissions in this lab class: 1.1 points.
* Submissions on time: 1 point.


In [14]:
import numpy as np
import time

def my_range(start, end):
    if start <= end:
        return range(start, end + 1)
    else:
        return range(start, end - 1, -1)


class Problem:
    char_mapping = ('·', 'Q')

    def __init__(self, n=4):
        self.N = n

    def is_valid(self, state):
        """
        check whether the state satisfies the condition or not.
        : param state: list of points (in (row id, col id) tuple form)
        : return: bool value of valid or not
        """
        board = self.get_board(state)
        res = True
        for point in state:
            i, j = point
            condition1 = board[:, j].sum() <= 1
            condition2 = board[i, :].sum() <= 1
            condition3 = self.pos_slant_condition(board, point)
            condition4 = self.neg_slant_condition(board, point)
            res = res and condition1 and condition2 and condition3 and condition4
            if not res:
                break
        return res

    def is_satisfy(self, state):
        return self.is_valid(state) and len(state) == self.N

    def pos_slant_condition(self, board, point):
        i, j = point
        tmp = min(self.N - i - 1, j)
        start = (i + tmp, j - tmp)
        tmp = min(i, self.N - j - 1)
        end = (i - tmp,  j + tmp)
        rows = my_range(start[0], end[0])
        cols = my_range(start[1], end[1])
        return board[rows, cols].sum() <= 1

    def neg_slant_condition(self, board, point):
        i, j = point
        tmp = min(i, j)
        start = (i - tmp, j - tmp)
        tmp = min(self.N - i - 1, self.N - j - 1)
        end = (i + tmp, j + tmp)
        rows = my_range(start[0], end[0])
        cols = my_range(start[1], end[1])
        return board[rows, cols].sum() <= 1

    def get_board(self, state):
        board = np.zeros([self.N, self.N], dtype=int)
        for point in state:
            board[point] = 1
        return board

    def print_state(self, state):
        board = self.get_board(state)
        print('_' * (2 * self.N + 1))
        for row in board:
            for item in row:
                print(f'|{Problem.char_mapping[item]}', end='')
            print('|')
        print('-' * (2 * self.N + 1))

In [15]:
class Solver:
    def __init__(self, n, method):
        self.N = n
        self.problem = Problem(n)
        self.method = method

    def solve(self, render=True):
        if render:
            import pygame
            w, h = 90 * self.n + 10, 90 * self.N + 10
            screen = pygame.display.set_mode((w, h))
            screen.fill('white')
            action_generator = self.method(self.problem)
            clk = pygame.time.Clock()
            queen_img = pygame.image.load('./queen.png')
            while True:
                for event in pygame.event.get():
                    if event == pygame.QUIT:
                        exit()
                try:
                    actions = next(action_generator)
                    screen.fill('white')
                    for i in range(self.n + 1):
                        pygame.draw.rect(screen, 'black', (i * 90, 0, 10, h))
                        pygame.draw.rect(screen, 'black', (0, i * 90, w, 10))
                    for action in actions:
                        i, j = action
                        screen.blit(queen_img, (10 + 90 * j, 10 + 90 * i))
                    pygame.display.flip()
                except StopIteration:
                    pass
                clk.tick(5)
            pass
        else:
            start_time = time.time()
            for actions in self.method(self.problem):
                pass
            self.problem.print_state(actions)
            print(time.time() - start_time)

# BTS: Backtracking search

In [16]:
def bts(problem):
    action_stack = [] #目前已放入的所有皇后位置

    #栈顶元素（m，n）表示第m行最后尝试的是放在n列
    stack = [(0, -1)]  # 从第0行开始尝试

    while not problem.is_satisfy(action_stack):
            
        row, col = stack.pop()
        
        next_col = col + 1
        found_valid_pos = False
        
        while next_col < problem.N:

            current_pos = (row, next_col) 
            
            temp_state = action_stack + [current_pos] #加入进来测试
            
            if problem.is_valid(temp_state):
                found_valid_pos = True
                
                stack.append((row, next_col))
                
                # Update action_stack
                action_stack.append(current_pos)
                yield action_stack
                
                # 这一行皇后处理完成，尝试放下一个皇后
                if row + 1 < problem.N:
                    stack.append((row + 1, -1))
                break
            
            next_col += 1 #当前位置皇后invalid，尝试放在下一个位置
            
        if not found_valid_pos: # 这一行没有位置valid，回溯
            if action_stack:
                action_stack.pop()
                yield action_stack

# Improving BTS 
* Which variable should be assigned next?
* In what order should its values be tried?
* Can we detect inevitable failure early?

## Minimum Remaining Value
Choose the variable with the fewest legal values in its domain
## Least constraining value
Given a variable, choose the least constraining value: the one that rules out the fewest values in the remaining variables
## Forward checking
Keep track of remaining legal values for the unassigned variables. Terminate when any variable has no legal values.

In [17]:
import copy

def improving_bts(problem):
    action_stack = []
    domains = {i: list(range(problem.N)) for i in range(problem.N)}  # 每行的可选列  
    untested_rows=[i for i in range(problem.N)] #未放置皇后的行
    stack = [([],domains,untested_rows)]  # 当前状态，目前合法的位置，未测试过的行

    while stack:
        
        cur_state, cur_domain, cur_untested_rows = stack.pop()

        while len(action_stack) > len(cur_state): # 说明回溯了，把action_stack弹出直到同个状态
            action_stack.pop()
            yield action_stack  
            
        # 当前快照有新进展，则将其加入 action_stack
        if len(cur_state) > len(action_stack):
            action_stack.append(cur_state[-1])
            yield action_stack

        if problem.is_satisfy(cur_state): #满足条件，退出
            break
        
        row = min(cur_untested_rows, key=lambda r: len(cur_domain[r])) #取出目前未测试行中合法位置最少者：MRV
            
        def count_conflicts(col):
            # 如果在这个(row, col)放皇后，会减少其余行多少个可选位置
            conflicts = 0
            for r in cur_untested_rows:
                if r != row:
                    for c in cur_domain[r]:
                        if c == col or (r - c == row - col) or (r + c == row + col): 
                            conflicts += 1
            return conflicts


        
                # 遍历（行，可选列），按其中对domains影响对列降序排序：LCV
        cols_to_try = sorted(cur_domain[row], key=count_conflicts, reverse=True)
        # 将合法位置从影响大到小进行压栈
        for col in cols_to_try:
            # 准备下一层的快照
            next_state = cur_state + [(row, col)]
            next_untested = [r for r in cur_untested_rows if r != row]
            next_domain = copy.deepcopy(cur_domain) 
            
            # 目前的state都是合法的，无需再次检查
            is_valid_branch = True
            for r in next_untested:
                # 过滤掉所有与当前(row, col)冲突的位置
                next_domain[r] = [
                    c for c in next_domain[r] 
                    if not (c == col or (r - c == row - col) or (r + c == row + col))
                ]
                # 再次剪枝:forward checking
                if not next_domain[r]:
                    is_valid_branch = False
                    break
                    
            if is_valid_branch:
                # 每个合法的col_to_try都压栈，用于后续回溯
                stack.append((next_state, next_domain, next_untested))

## Local search for CSP(min-conflict algorithm)

* First generate a complete assignment for all variables (this set of assignments may conflict)

* Repeat the following steps until there are no conflicts:

    * Randomly Select a variable that causes conflicts

    * Reassign the value of this variable to another value that with the least constraint onflicts with other variables

In [18]:
import random

def min_conflict(problem):
    action_stack = [(i, np.random.choice(problem.N)) for i in range(problem.N)]
    max_step = 1000
    cur_step = 0

    def is_conflict(r1, c1, r2, c2):
        return c1 == c2 or (r1 - c1 == r2 - c2) or (r1 + c1 == r2 + c2)
    
    while not problem.is_satisfy(action_stack) and cur_step < max_step:
        cur_step += 1
        conflicts = [] #记录当前有冲突的皇后位置
        for i1, c1 in action_stack:
            for i2, c2 in action_stack:
                if i1 != i2: #不和原来的自己比较
                    if is_conflict(i1, c1, i2, c2):
                        conflicts.append((i1, c1))
                        break
        if not conflicts: 
            break
        (x,y)=random.choice(conflicts)  #随机选择一个冲突的皇后

        #找出冲突最少的位置
        min_conflict_col = []  
        min_conflict_count = float('inf')
        for ny in range(problem.N):  
            if ny != y:
                conflict_count = 0
                for i, j in action_stack:
                    if i != x : #不和原来的自己比较
                        if is_conflict(x, ny, i, j):
                            conflict_count += 1
                if conflict_count < min_conflict_count:
                    min_conflict_col = [ny]
                    min_conflict_count = conflict_count
                elif conflict_count == min_conflict_count:
                    min_conflict_col.append(ny)
                    
        action_stack[x] = (x, random.choice(min_conflict_col))
        yield action_stack

## test block

In [19]:
Solver(n=6, method=bts).solve(render=False)

_____________
|·|Q|·|·|·|·|
|·|·|·|Q|·|·|
|·|·|·|·|·|Q|
|Q|·|·|·|·|·|
|·|·|Q|·|·|·|
|·|·|·|·|Q|·|
-------------
0.007986783981323242


In [20]:
Solver(n=16, method=improving_bts).solve(render=False)

_________________________________
|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|Q|
|·|·|·|·|·|·|·|·|·|·|·|·|Q|·|·|·|
|·|·|·|·|·|·|·|·|·|Q|·|·|·|·|·|·|
|·|·|·|·|·|·|Q|·|·|·|·|·|·|·|·|·|
|·|·|·|·|Q|·|·|·|·|·|·|·|·|·|·|·|
|·|·|·|·|·|Q|·|·|·|·|·|·|·|·|·|·|
|·|·|·|·|·|·|·|·|·|·|·|·|·|·|Q|·|
|Q|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|
|·|·|Q|·|·|·|·|·|·|·|·|·|·|·|·|·|
|·|Q|·|·|·|·|·|·|·|·|·|·|·|·|·|·|
|·|·|·|·|·|·|·|·|Q|·|·|·|·|·|·|·|
|·|·|·|·|·|Q|·|·|·|·|·|·|·|·|·|·|
|·|·|Q|·|·|·|·|·|·|·|·|·|·|·|·|·|
|·|·|·|·|·|·|·|·|·|·|Q|·|·|·|·|·|
|·|·|·|Q|·|·|·|·|·|·|·|·|·|·|·|·|
|·|·|·|·|·|·|·|·|·|·|·|Q|·|·|·|·|
---------------------------------
0.06696009635925293


In [21]:
Solver(n=100, method=min_conflict).solve(render=False)

_________________________________________________________________________________________________________________________________________________________________________________________________________
|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|Q|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|
|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|Q|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|
|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|Q|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|
|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|Q|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·|·